# Lab 2.4 — Pruning, Conversion, and Deployment to Raspberry Pi Pico

This notebook has `# TODO` markers (TODO 1-8). Work through them in order.

Copy your `mnist_cnn.h5` from Lab 2.2 into this same folder before you start.

In [ ]:
import os
import time
import gzip
import numpy as np
import pandas as pd
import tensorflow as tf
import tf_keras
import tensorflow_model_optimization as tfmot
import matplotlib.pyplot as plt

tf.random.set_seed(10)
np.random.seed(10)

print("TensorFlow version:", tf.__version__)

## Step 0 — Load data and the Lab 2.2 model

This part is provided for you.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train_n = (x_train / 255.0).astype("float32")[..., np.newaxis]
x_test_n  = (x_test  / 255.0).astype("float32")[..., np.newaxis]

model = tf_keras.models.load_model("mnist_cnn.h5")
_, baseline_acc = model.evaluate(x_test_n, y_test, verbose=0)
baseline_size_kb = os.path.getsize("mnist_cnn.h5") / 1024
print(f"Float32 baseline -- size: {baseline_size_kb:.1f} KB, accuracy: {baseline_acc:.4f}")

## Part A — Pruning

`prune_low_magnitude` wraps every prunable layer so that, during training, the
smallest-magnitude weights in each layer are progressively zeroed out
following a schedule.

**TODO 1:** Build a `PolynomialDecay` pruning schedule: start at 0% sparsity,
ramp to a **50% final sparsity target**, over `end_step` steps (already
computed for you below — `steps_per_epoch * epochs`). Then wrap `model` with
`tfmot.sparsity.keras.prune_low_magnitude()` using that schedule, and
compile it (same loss/metric as always).

In [ ]:
num_images = x_train_n.shape[0]
batch_size = 64
prune_epochs = 2
end_step = int(np.ceil(num_images / batch_size)) * prune_epochs

# TODO 1: build pruning_params with a PolynomialDecay schedule (0% -> 50%
# sparsity over end_step steps), wrap `model` with prune_low_magnitude(),
# and compile the result.
pruned_model = None

pruned_model.summary()

**TODO 2:** Fine-tune `pruned_model`. This **requires** the
`tfmot.sparsity.keras.UpdatePruningStep()` callback — it's what actually
advances the pruning schedule each step. Skip it and the model trains
normally but never actually gets pruned. Use `prune_epochs` from above.

In [ ]:
# TODO 2: fit pruned_model with the UpdatePruningStep callback


**TODO 3:** Strip the pruning wrappers with
`tfmot.sparsity.keras.strip_pruning()`, then compile and evaluate the
stripped model. **This step is mandatory** — `prune_low_magnitude` wraps each
layer in a `PruneLowMagnitude` wrapper that only exists for training-time
bookkeeping; the model won't convert to TFLite cleanly with those wrappers
still attached.

Also check the actual sparsity achieved (a loop over `trainable_weights`
checking `np.mean(w.numpy() == 0)` for kernel weights is provided below) —
confirm it's close to your 50% target.

In [ ]:
# TODO 3: strip pruning wrappers, compile, evaluate, and save as pruned_model.h5
stripped_model = None
pruned_acc = None
pruned_size_kb = None

print(f"Pruned (pre-quantization) -- size: {pruned_size_kb:.1f} KB, accuracy: {pruned_acc:.4f}")

for w in stripped_model.trainable_weights:
    if "kernel" in w.name:
        sparsity = np.mean(w.numpy() == 0)
        print(f"  {w.name}: {sparsity:.1%} zeros")

## Part B — Combine Pruning + Quantization

Same PTQ process as Lab 2.3, applied to the *pruned* model instead of the
original.

**TODO 4:** Configure a `TFLiteConverter` for full-integer int8 quantization
(same settings as Lab 2.3), convert `stripped_model`, and save as
`model.tflite`.

In [ ]:
def representative_dataset():
    for i in range(200):
        yield [x_train_n[i:i+1]]

# TODO 4: build and configure the converter, convert, and save model.tflite
tflite_pruned_quant = None

pq_size_kb = len(tflite_pruned_quant) / 1024
print(f"Pruned + quantized: {pq_size_kb:.1f} KB")

**TODO 5:** Evaluate the pruned+quantized TFLite model (reuse your
`evaluate_tflite()` pattern from Lab 2.3 — a fresh copy is provided below to
complete). Also print the input tensor's `(scale, zero_point)` — you'll see
these same values reflected in how the Pico C++ code quantizes images.

In [ ]:
def evaluate_tflite(tflite_bytes, x, y, n_latency=100):
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()[0]
    out = interpreter.get_output_details()[0]
    scale, zero_point = inp["quantization"]

    def quantize(img):
        return (img / scale + zero_point).round().astype(np.int8)

    # TODO 5a: loop over all of x/y, run inference, count correct predictions
    correct = 0
    accuracy = None

    # TODO 5b: time n_latency invocations (same pattern as Lab 2.3)
    latency_ms = None

    return accuracy, latency_ms, inp["dtype"], out["dtype"], scale, zero_point

pq_acc, pq_latency_ms, pq_in_dtype, pq_out_dtype, input_scale, input_zero_point = \
    evaluate_tflite(tflite_pruned_quant, x_test_n, y_test)

print(f"Pruned + quantized -- accuracy: {pq_acc:.4f}, latency: {pq_latency_ms:.3f} ms/image, "
      f"input dtype: {pq_in_dtype}, output dtype: {pq_out_dtype}")
print(f"Input quantization -- scale: {input_scale}, zero_point: {input_zero_point}")

**Does pruning + quantization beat quantization alone?** Compare the raw
`.tflite` file size against Lab 2.3's PTQ-only number. TFLite stores pruned
weights as literal zero bytes in a dense array, so don't assume pruning
automatically shrinks the raw flatbuffer — check what it actually does to
size, and what happens once the file is compressed (gzip).

**A trap worth knowing about:** `prune_low_magnitude()` wraps the *same*
underlying layer objects, by reference, not copies. Fine-tuning
`pruned_model` in TODO 2 therefore also mutated `model`'s weights in place —
`model` is no longer the original float32 baseline. For a fair "unpruned"
comparison here, reload a fresh copy from `mnist_cnn.h5` rather than reusing
`model`.

In [ ]:
def gzip_size_kb(path):
    with open(path, "rb") as f:
        data = f.read()
    return len(gzip.compress(data, compresslevel=9)) / 1024

# TODO 6: reload a FRESH copy of mnist_cnn.h5 (don't reuse `model` -- see the
# note above about why), quantize it the same way as TODO 4, and save as
# model_unpruned_quant.tflite. Then compare raw and gzip-compressed sizes
# between the unpruned and pruned quantized models.
model_unpruned_fresh = None
tflite_unpruned_quant = None


## Final comparison across all 4 sessions

In [ ]:
comparison = pd.DataFrame([
    {"variant": "Float32 baseline (2.2)", "size_kb": round(baseline_size_kb, 1), "accuracy": round(baseline_acc, 4)},
    # Fill in your actual Lab 2.3 PTQ/QAT numbers here:
    {"variant": "PTQ int8 (2.3)", "size_kb": None, "accuracy": None},
    {"variant": "QAT int8 (2.3)", "size_kb": None, "accuracy": None},
    {"variant": "Pruned + quantized (2.4)", "size_kb": round(pq_size_kb, 1), "accuracy": round(pq_acc, 4)},
])
comparison

## Part C — Convert for Deployment

> **Note:** TensorFlow Lite has been rebranded **LiteRT**. The conversion API
> and file format are unchanged; only the name and some docs have moved.

`xxd -i` turns the `.tflite` flatbuffer into a C array, but the symbol names
it generates (`model_tflite`, `model_tflite_len`) don't match what
`pico_sw/src/mnist_model_data.h` declares (`mnist_model_data`,
`mnist_model_data_len`).

**TODO 7:** Run `xxd -i model.tflite` (via `subprocess`, captured as text),
then rename `model_tflite` -> `mnist_model_data` and `model_tflite_len` ->
`mnist_model_data_len` in the output. Also prepend
`#include "mnist_model_data.h"` and change `unsigned char` to
`alignas(8) const unsigned char` (TFLite Micro expects 8-byte-aligned model
data). Write the result to `mnist_model_data.cpp`.

In [ ]:
import subprocess

result = subprocess.run(["xxd", "-i", "model.tflite"], capture_output=True, text=True, check=True)
xxd_output = result.stdout

# TODO 7: transform xxd_output into the final C++ source and write it to
# mnist_model_data.cpp
c_array = None

with open("mnist_model_data.cpp", "w") as f:
    f.write(c_array)

print(f"Wrote mnist_model_data.cpp ({len(c_array)} chars)")

**TODO 8:** Sanity check — confirm `mnist_model_data_len` in your
generated file matches the actual `.tflite` file's byte size. A mismatch
here means the interpreter will read past the end of the array on-device.

In [ ]:
# TODO 8: print len(tflite_pruned_quant) and confirm it matches the
# mnist_model_data_len value written into mnist_model_data.cpp above


Copy `mnist_model_data.cpp` into `pico_sw/models/mnist_model_data.cpp`,
replacing the placeholder there (TODO 1/2 in that file).

## Deliverable checklist

- [ ] Pruned + quantized `.tflite` file (`model.tflite` above)
- [ ] Final comparison table across all 4 sessions (fill in your actual Lab
      2.3 numbers in the table above, plus your own-handwritten-digit and
      on-device accuracy once you've run Part D on the Pico)
- [ ] Short writeup: which combination would you actually ship on a
      resource-constrained microcontroller, and why?